In [1]:
import pandas as pd
import re
import emoji
from bertopic import BERTopic
from umap import UMAP
from sentence_transformers import SentenceTransformer

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

e:\Naad\Projects\Real-Time Social Media Monitoring of Rupiah Exchange Rate Issues Using Transformer-Based NLP\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('../data/processed/twitter_data_cleaned.csv')

In [3]:
df.head(10)

,id,text,retweetCount,replyCount,likeCount,quoteCount,viewCount,createdAt,bookmarkCount,author,tweet_length
0,2058239245148651767,kondisi negara udah bener bener diluar nalar\n...,0,0,0,0,2,2026-05-23,0,ensivo,230
1,2058239197652406671,ciecie joong jadi rupiah bgt didepan bininya 🤭...,0,0,0,0,2,2026-05-23,0,natachaintikk,71
2,2058239004710158506,membeli semua make up konser 0 rupiah karena c...,0,0,0,0,0,2026-05-23,0,3JSPACE,63
3,2058238886363668815,Nilai tukar rupiah gak disamain ma yen aja git...,0,1,0,0,2,2026-05-23,0,rainy_snowflake,122
4,2058238081271308426,aduh aku bacanya ikutan kaya rupiah,0,0,0,0,3,2026-05-23,0,zss0ur,35
5,2058237994013004231,Bukit Tinggi 2026 #Banser The largest moderate...,0,1,0,0,37,2026-05-23,0,wahabi92013848,280
6,2058237966309613886,gua mau ngomong kangen aja mikir sampe rupiah ...,0,0,0,0,5,2026-05-23,0,voulGare,70
7,2058237951377805472,@CNNIndonesia - Berapa Trilyun Rupiah yg sudah...,0,0,0,0,4,2026-05-23,0,meliksuman74092,207
8,2058237793986556403,untung beli membership sblm rupiah ekhem…,0,0,0,0,0,2026-05-23,0,Jinniejuiceyo,41
9,2058237313441030174,"@mahasmashana @muncorner Males, nggak ada yang...",0,0,0,0,12,2026-05-23,0,warmestarms,76


In [4]:
def preprocess(text):
    if pd.isna(text):
        return ""

    text=text.lower() # case folding
    text=re.sub(r"http\S+"," ",text) # url
    text=re.sub(r"@\w+"," ",text) # mention
    text=re.sub(r"#"," ",text) # hashtag
    text=re.sub(r"\brt\b"," ",text) # RT
    text=emoji.replace_emoji(text,'') # emoji
    text=re.sub(r"[^a-zA-Z0-9\s]"," ",text) # simbol
    text=re.sub(r"\s+"," ",text).strip() # spasi berlebih
    text = re.sub(r"\b\d+\b","",text)

    return text

df["clean_text"] = df['text'].apply(preprocess)

### Normalisasi

In [5]:
norm={
    " gk ":" tidak ",
    " ga ":" tidak ",
    " gak ":" tidak ",
    " yg ":" yang ",
    " dpt ":" dapat ",
    " bgt ":" banget ",
    " krn ":" karena ",
    " udh ":" sudah ",
    " tdk ":" tidak ",
    " gt ":" gitu ",
    " klo ":" kalau ",
    " kalo ":" kalau ",
    " aja ":" saja ",
    " nih ":" ini ",
    " ama ":" sama ",
    " ma ":" sama ",
    " siaapp ":" siap ",
    " bnr ":" benar ",
    " gpp ":" tidak apa-apa ",
    " bener ":" benar ",
    " dr ":" dari ",
    " hrs ":" harus ",
    " nnti ":" nanti ",
    " nnt ":" nanti ",
    " dgn ":" dengan ",
    " sm ":" sama ",
    " sy ":" saya ",
    " lg ":" lagi ",
    " hrsnya ":" seharusnya ",
    " hrsnha ":" seharusnya ",
    " hrsnyaa ":" seharusnya ",
    " hrsnyaaa ":" seharusnya ",
    " sblm ":" sebelum ",
    " sblmnya ":" sebelumnya ",
    " bgttt ":" banget ",
    " tpi ":" tapi ",
    " tp ":" tapi ",
    " bgtu ":" begitu ",
    " blm ":" belum ",
    " gmn ":" gimana ",
    " mls ":" malas ",
    "tp ":" tapi ",
    "tpi ":" tapi ",
    " kpd ":" kepada ",
    " brp ":" berapa ",
    " bs ":" bisa ",
    " mo ":" mau ",
    " krj ":" kerja "
}

def normalisasi(str_text):
  for i in norm:
    str_text = str_text.replace(i, norm[i])
  return str_text

df['clean_text'] = df['clean_text'].apply(lambda x: normalisasi(x))

In [6]:
df = df[df["clean_text"].str.strip()!=""]

In [ ]:
df.drop_duplicates(subset=["clean_text"],inplace=True)
df.dropna()

### Stopwords

In [8]:
factory = StopWordRemoverFactory()
stop_words = set(factory.get_stop_words())

more_stop_words={
    "rt",
    "amp",
    "gue",
    "gw",
    "gua",
    "wkwk",
    "wk",
    "wkwkwk",
    "haha",
    "hehe",
    "lol",
    "bro",
    "min",
    "nya"
}
stop_words.update(more_stop_words)


def remove_stopwords(text):
    words = text.split()
    filtered_words = [
        word.lower()
        for word in words
        if word.lower() not in stop_words
    ]
    return " ".join(filtered_words)


df["clean_text"] = (df["clean_text"].astype(str).apply(remove_stopwords))

In [9]:
df.head()

,id,retweetCount,replyCount,likeCount,quoteCount,viewCount,createdAt,bookmarkCount,author,tweet_length,clean_text
0,2058239245148651767,0,0,0,0,2,2026-05-23,0,ensivo,230,kondisi negara udah benar bener diluar nalar s...
1,2058239197652406671,0,0,0,0,2,2026-05-23,0,natachaintikk,71,ciecie joong jadi rupiah banget didepan bininya
2,2058239004710158506,0,0,0,0,0,2026-05-23,0,3JSPACE,63,membeli semua make up konser rupiah co pake so...
3,2058238886363668815,0,1,0,0,2,2026-05-23,0,rainy_snowflake,122,nilai tukar rupiah disamain sama yen gitu biar...
4,2058238081271308426,0,0,0,0,3,2026-05-23,0,zss0ur,35,aduh aku bacanya ikutan kaya rupiah


### Topic Modeling

In [10]:
spam_words = [
    "download",
    "saldo",
    "promo",
    "shopeepay",
    "voucher",
    "zimbabwe",
    "world",
    "largest"
]

pattern = "|".join(spam_words)

df = df[
    ~df["clean_text"]
    .str.contains(
        pattern,
        case=False,
        na=False
    )
]

df = df.reset_index(drop=True)

In [11]:
documents = (
    df["clean_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

In [12]:
embedding_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    language="multilingual",
    min_topic_size=15,
    nr_topics=None,
    calculate_probabilities=True,
    verbose=True
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1084.75it/s]


In [13]:
topics, probs = topic_model.fit_transform(documents)

2026-05-25 14:17:13,564 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 265/265 [07:04<00:00,  1.60s/it]
2026-05-25 14:24:18,355 - BERTopic - Embedding - Completed ✓
2026-05-25 14:24:18,358 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-25 14:26:09,988 - BERTopic - Dimensionality - Completed ✓
2026-05-25 14:26:09,993 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-25 14:26:13,975 - BERTopic - Cluster - Completed ✓
2026-05-25 14:26:14,101 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-25 14:26:15,077 - BERTopic - Representation - Completed ✓


In [14]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1404,-1_rupiah_naik_dollar_krisis,"[rupiah, naik, dollar, krisis, ekonomi, kalau,...","[gini kalau uang rupiah melemah, harganya naik..."
1,0,1889,0_indonesia_ekonomi_prabowo_nasional,"[indonesia, ekonomi, prabowo, nasional, presid...",[bank indonesia menaikkan bi rate menjadi lang...
2,1,1630,1_inflasi_harga_naik_desa,"[inflasi, harga, naik, desa, dollar, kalau, ru...","[sekarang inflasi, inflasi naik, inflasi]"
3,2,990,2_dollar_naik_makin_banget,"[dollar, naik, makin, banget, kalau, harga, te...","[naik terus cuma dollar, dollar naik, dollar n..."
4,3,773,3_rupiah_aku_lemah_melemah,"[rupiah, aku, lemah, melemah, banget, mau, lu,...","[kalau rupiah melemah, rupiah lemah, lemah rup..."
5,4,689,4_krisis_ekonomi_negara_bukan,"[krisis, ekonomi, negara, bukan, rakyat, kalau...","[penting krisis ekonomi kak, betul jadi krisis..."
6,5,147,5_rupiah_juta_berapa_ribu,"[rupiah, juta, berapa, ribu, gede, kalau, doll...",[juta rupiah berasa juta dollar kali dibilang ...
7,6,133,6_bekalan_global_krisis_ekonomi,"[bekalan, global, krisis, ekonomi, kerajaan, m...",[taklimat krisis bekalan global menteri ekonom...
8,7,133,7_presiden_dollar_naik_desa,"[presiden, dollar, naik, desa, orang, bilang, ...",[biar mikir tuh presiden beras plastik makanan...
9,8,76,8_malaysia_bekalan_krisis_global,"[malaysia, bekalan, krisis, global, ekonomi, n...",[ekonomi malaysia terus kukuh sebalik krisis g...


In [15]:
df["topic"]=topics

In [17]:
topic_model.visualize_barchart()

Pelemahan rupiah tidak hanya memunculkan percakapan terkait kurs mata uang, tetapi juga memicu diskusi mengenai inflasi, kebijakan pemerintah, serta potensi krisis ekonomi.

Visualisasi topic word score menunjukkan kata-kata dengan nilai c-TF-IDF tertinggi pada setiap topik. Nilai yang lebih tinggi menunjukkan bahwa suatu kata lebih representatif terhadap topik tertentu dibandingkan topik lainnya. Hasil visualisasi memperlihatkan dominasi kata seperti "rupiah", "dollar", "inflasi", dan "krisis", yang mengindikasikan fokus utama percakapan publik terkait pelemahan nilai tukar rupiah.

In [18]:
topic_model.visualize_hierarchy()

## Topic Mapping
| Cluster   |                                Macro Topic | Topic ID                       | Interpretasi                                                                                                                        |
| --------- | -----------------------------------------: | ------------------------------ | ----------------------------------------------------------------------------------------------------------------------------------- |
| Cluster 1 |                     Global Economic Crisis | 8, 6                           | Percakapan mengenai krisis ekonomi global dan pengaruh kondisi internasional terhadap ekonomi domestik.                             |
| Cluster 2 | Macroeconomic Crisis & Government Response | 13, 12, 4, 0, 10, 21           | Percakapan terkait krisis ekonomi, krisis moneter, kebijakan pemerintah, serta respons institusi ekonomi.                           |
| Cluster 3 |                   Rupiah Exchange Dynamics | 18, 20, 5, 3, 15               | Percakapan mengenai pelemahan rupiah, nilai tukar, penurunan nilai mata uang, dan dampak langsung kurs.                             |
| Cluster 4 |             Food Price & Inflation Control | 11, 24, 19                     | Percakapan terkait harga pangan, pengendalian inflasi, dan kebutuhan pokok masyarakat.                                              |
| Cluster 5 |         Inflation & Public Economic Impact | 22, 9, 26, 7, 2, 1, 17, 14, 16 | Percakapan mengenai inflasi, kenaikan harga, dampak terhadap masyarakat, pekerjaan, serta persepsi publik terhadap kondisi ekonomi. |
| Cluster 6 |          Financial Market & Digital Assets | 25, 23                         | Percakapan mengenai aset digital, cryptocurrency, dan pasar keuangan alternatif.                                                    |

# Hasil
Hasil hierarchical clustering mengelompokkan 27 topik hasil BERTopic ke dalam enam tema makro. Tema utama terdiri atas krisis ekonomi global, krisis ekonomi dan respons pemerintah, dinamika nilai tukar rupiah, harga pangan dan pengendalian inflasi, dampak inflasi terhadap masyarakat, serta pasar keuangan digital. Kelompok dengan volume tertinggi menunjukkan dominasi percakapan mengenai dampak ekonomi publik dan pelemahan nilai tukar rupiah.

In [19]:
df

,id,retweetCount,replyCount,likeCount,quoteCount,viewCount,createdAt,bookmarkCount,author,tweet_length,clean_text,topic
0,2058239245148651767,0,0,0,0,2,2026-05-23,0,ensivo,230,kondisi negara udah benar bener diluar nalar s...,0
1,2058239197652406671,0,0,0,0,2,2026-05-23,0,natachaintikk,71,ciecie joong jadi rupiah banget didepan bininya,3
2,2058239004710158506,0,0,0,0,0,2026-05-23,0,3JSPACE,63,membeli semua make up konser rupiah co pake so...,9
3,2058238886363668815,0,1,0,0,2,2026-05-23,0,rainy_snowflake,122,nilai tukar rupiah disamain sama yen gitu biar...,-1
4,2058238081271308426,0,0,0,0,3,2026-05-23,0,zss0ur,35,aduh aku bacanya ikutan kaya rupiah,3
...,...,...,...,...,...,...,...,...,...,...,...,...
8449,2057009597882544262,0,0,0,0,133,2026-05-20,0,vier4x4,313,makanya banyak bingung beliau bilang masyaraka...,0
8450,2057009570858557495,0,0,0,0,19,2026-05-20,0,PohanChoky40724,304,vietnam luka perang menjadi macan ekonomi asia...,0
8451,2057008966241247646,0,0,0,0,9,2026-05-20,0,balita655,299,serangan serangan datang media asing sebut bor...,0
8452,2057008504138051652,0,0,0,0,33,2026-05-20,0,antaranews_bali,125,purbaya beberkan alasan ogah naikkan harga bbm...,0


In [20]:
df.to_csv("../data/processed/topic_data.csv",index=False)